# Análise Exploratória da Base de Dados do Sinesp VDE 2024
Este notebook tem como objetivo explorar e tratar a base de dados referente aos crimes ocorridos em 2024 nos estados do Nordeste e Sudeste. Esses registros, posteriormente, serão cruzados com os dados de infraestrutura educacional e taxas de pobreza, com o intuito de gerar informações sobre a correlação entre violência e vulnerabilidade social.

A fonte de dados foi retirada do site do governo: https://www.gov.br/mj/pt-br/assuntos/sua-seguranca/seguranca-publica/estatistica/dados-nacionais-1/base-de-dados-e-notas-metodologicas-dos-gestores-estaduais-sinesp-vde-2022-e-2023

In [2]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

In [3]:
from pathlib import Path

current_dir = Path.cwd()
root = current_dir.parent
data_path = root / 'data' / 'raw' / 'crimes_2024.csv'

In [4]:
df_crimes = pd.read_csv(data_path, sep=';')
df_crimes.head()

,uf,municipio,evento,data_referencia,agente,arma,faixa_etaria,feminino,masculino,nao_informado,total_vitima,total,total_peso,abrangencia
0,AC,ACRELÂNDIA,Feminicídio,1/1/2024,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,Estadual
1,AC,ACRELÂNDIA,Feminicídio,1/2/2024,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,Estadual
2,AC,ACRELÂNDIA,Feminicídio,1/3/2024,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,Estadual
3,AC,ACRELÂNDIA,Feminicídio,1/4/2024,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,Estadual
4,AC,ACRELÂNDIA,Feminicídio,1/5/2024,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,Estadual


In [5]:
df_crimes = df_crimes.drop(columns=['agente','arma','faixa_etaria','feminino','masculino','nao_informado','total_peso'])
df_crimes.head()

,uf,municipio,evento,data_referencia,total_vitima,total,abrangencia
0,AC,ACRELÂNDIA,Feminicídio,1/1/2024,0.0,NaN,Estadual
1,AC,ACRELÂNDIA,Feminicídio,1/2/2024,0.0,NaN,Estadual
2,AC,ACRELÂNDIA,Feminicídio,1/3/2024,0.0,NaN,Estadual
3,AC,ACRELÂNDIA,Feminicídio,1/4/2024,0.0,NaN,Estadual
4,AC,ACRELÂNDIA,Feminicídio,1/5/2024,0.0,NaN,Estadual


In [6]:
df_crimes.loc[:, 'data_referencia'] = pd.to_datetime(df_crimes['data_referencia'])
df_crimes['mes'] = pd.DatetimeIndex(df_crimes['data_referencia']).month
df_crimes.drop(columns='data_referencia')

,uf,municipio,evento,total_vitima,total,abrangencia,mes
0,AC,ACRELÂNDIA,Feminicídio,0.0,NaN,Estadual,1
1,AC,ACRELÂNDIA,Feminicídio,0.0,NaN,Estadual,1
2,AC,ACRELÂNDIA,Feminicídio,0.0,NaN,Estadual,1
3,AC,ACRELÂNDIA,Feminicídio,0.0,NaN,Estadual,1
4,AC,ACRELÂNDIA,Feminicídio,0.0,NaN,Estadual,1
...,...,...,...,...,...,...,...
764783,TO,XAMBIOÁ,Tentativa de homicídio,0.0,NaN,Estadual,1
764784,TO,XAMBIOÁ,Tentativa de homicídio,0.0,NaN,Estadual,1
764785,TO,XAMBIOÁ,Tentativa de homicídio,2.0,NaN,Estadual,1
764786,TO,XAMBIOÁ,Tentativa de homicídio,1.0,NaN,Estadual,1


In [7]:
df_crimes = df_crimes.rename(columns={"total":"ocorrencia"})
df_crimes.head()

,uf,municipio,evento,data_referencia,total_vitima,ocorrencia,abrangencia,mes
0,AC,ACRELÂNDIA,Feminicídio,2024-01-01 00:00:00,0.0,NaN,Estadual,1
1,AC,ACRELÂNDIA,Feminicídio,2024-01-02 00:00:00,0.0,NaN,Estadual,1
2,AC,ACRELÂNDIA,Feminicídio,2024-01-03 00:00:00,0.0,NaN,Estadual,1
3,AC,ACRELÂNDIA,Feminicídio,2024-01-04 00:00:00,0.0,NaN,Estadual,1
4,AC,ACRELÂNDIA,Feminicídio,2024-01-05 00:00:00,0.0,NaN,Estadual,1


In [8]:
uf_nordeste_sudeste = ['AL','BA','CE','MA','PB','PE','PI','RN','SE','ES','MG','RJ','SP']
df_crimes['uf'] = df_crimes['uf'].str.upper()
df_crimes = df_crimes[df_crimes['uf'].isin(uf_nordeste_sudeste)]
df_crimes

,uf,municipio,evento,data_referencia,total_vitima,ocorrencia,abrangencia,mes
3828,AL,ANADIA,Feminicídio,2024-01-01 00:00:00,0.0,NaN,Estadual,1
3829,AL,ANADIA,Feminicídio,2024-01-02 00:00:00,0.0,NaN,Estadual,1
3830,AL,ANADIA,Feminicídio,2024-01-03 00:00:00,0.0,NaN,Estadual,1
3831,AL,ANADIA,Feminicídio,2024-01-04 00:00:00,0.0,NaN,Estadual,1
3832,AL,ANADIA,Feminicídio,2024-01-05 00:00:00,0.0,NaN,Estadual,1
...,...,...,...,...,...,...,...,...
745511,SP,ÓLEO,Tentativa de homicídio,2024-01-08 00:00:00,1.0,NaN,Estadual,1
745512,SP,ÓLEO,Tentativa de homicídio,2024-01-09 00:00:00,1.0,NaN,Estadual,1
745513,SP,ÓLEO,Tentativa de homicídio,2024-01-10 00:00:00,0.0,NaN,Estadual,1
745514,SP,ÓLEO,Tentativa de homicídio,2024-01-11 00:00:00,0.0,NaN,Estadual,1


In [9]:
pd.set_option('display.max_colwidth', None)
print(df_crimes['evento'].drop_duplicates())

3828                                                            Feminicídio
3840                                                       Homicídio doloso
3852                                        Lesão corporal seguida de morte
3864                                             Mandado de prisão cumprido
3876     Morte no trânsito ou em decorrência dele (exceto homicídio doloso)
3888                             Mortes a esclarecer (sem indício de crime)
3900                                                     Mortes no trânsito
3912                                    Roubo seguido de morte (latrocínio)
3924                                                               Suicídio
3936                                               Tentativa de feminicídio
3948                                                 Tentativa de homicídio
11616                                                  Apreensão de Cocaína
11640                                                  Apreensão de Maconha
11664       

### Seleção dos eventos
Ao analisarmos os eventos presentes nesta base de dados, verificamos que existem registros que não se classificam como crime culposo. Além disso, não temos apenas delitos mas também eventos de caráter administrativo. Como esses registros não são o foco deste projeto, foram retirados do dataset.

In [10]:
df_crimes['evento'] = df_crimes['evento'].str.strip()
drop_events = ['Mandado de prisão cumprido', 'Morte no trânsito ou em decorrência dele (exceto homicídio doloso)', 'Mortes a esclarecer (sem indício de crime)','Mortes no trânsito','Suicídio','Atendimento pré-hospitalar','Busca e salvamento','Combate a incêndios','Emissão de Alvarás de licença','Morte de Agente do Estado','Morte por intervenção de Agente do Estado','Pessoa Desaparecida','Pessoa Localizada','Realização de vistorias','Suicídio de Agente do Estado']
df_crimes = df_crimes[~df_crimes['evento'].isin(drop_events)]

In [11]:
print(df_crimes['evento'].drop_duplicates())

3828                             Feminicídio
3840                        Homicídio doloso
3852         Lesão corporal seguida de morte
3912     Roubo seguido de morte (latrocínio)
3936                Tentativa de feminicídio
3948                  Tentativa de homicídio
11616                   Apreensão de Cocaína
11640                   Apreensão de Maconha
11664                Arma de Fogo Apreendida
11928                                Estupro
11940                  Estupro de vulnerável
11964                       Furto de veículo
12288         Roubo a instituição financeira
12312                         Roubo de carga
12336                       Roubo de veículo
12516                      Tráfico de drogas
Name: evento, dtype: object


In [12]:
df_crimes.info()

<class 'pandas.core.frame.DataFrame'>
Index: 255624 entries, 3828 to 745515
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   uf               255624 non-null  object 
 1   municipio        255624 non-null  object 
 2   evento           255624 non-null  object 
 3   data_referencia  255624 non-null  object 
 4   total_vitima     250632 non-null  float64
 5   ocorrencia       3354 non-null    float64
 6   abrangencia      255624 non-null  object 
 7   mes              255624 non-null  int32  
dtypes: float64(2), int32(1), object(5)
memory usage: 16.6+ MB


In [13]:
print(df_crimes.isnull().sum())

uf                      0
municipio               0
evento                  0
data_referencia         0
total_vitima         4992
ocorrencia         252270
abrangencia             0
mes                     0
dtype: int64


In [14]:
print(df_crimes['abrangencia'].drop_duplicates())

3828            Estadual
11616    Polícia Federal
Name: abrangencia, dtype: object


In [15]:
abrangencia_policia = df_crimes[(df_crimes['abrangencia'] == 'Polícia Federal')]
abrangencia_policia

,uf,municipio,evento,data_referencia,total_vitima,ocorrencia,abrangencia,mes
11616,AL,NÃO INFORMADO,Apreensão de Cocaína,2024-01-01 00:00:00,NaN,NaN,Polícia Federal,1
11619,AL,NÃO INFORMADO,Apreensão de Cocaína,2024-01-02 00:00:00,NaN,NaN,Polícia Federal,1
11621,AL,NÃO INFORMADO,Apreensão de Cocaína,2024-01-03 00:00:00,NaN,NaN,Polícia Federal,1
11623,AL,NÃO INFORMADO,Apreensão de Cocaína,2024-01-04 00:00:00,NaN,NaN,Polícia Federal,1
11624,AL,NÃO INFORMADO,Apreensão de Cocaína,2024-01-05 00:00:00,NaN,NaN,Polícia Federal,1
...,...,...,...,...,...,...,...,...
709867,SP,NÃO INFORMADO,Tráfico de drogas,2024-01-08 00:00:00,NaN,133.0,Polícia Federal,1
709868,SP,NÃO INFORMADO,Tráfico de drogas,2024-01-09 00:00:00,NaN,116.0,Polícia Federal,1
709871,SP,NÃO INFORMADO,Tráfico de drogas,2024-01-10 00:00:00,NaN,146.0,Polícia Federal,1
709873,SP,NÃO INFORMADO,Tráfico de drogas,2024-01-11 00:00:00,NaN,98.0,Polícia Federal,1


In [16]:
abrangencia_policia['municipio'].drop_duplicates()

11616    NÃO INFORMADO
Name: municipio, dtype: object

In [17]:
abrangencia_estadual = df_crimes[(df_crimes['abrangencia'] == 'Estadual')]
abrangencia_estadual

,uf,municipio,evento,data_referencia,total_vitima,ocorrencia,abrangencia,mes
3828,AL,ANADIA,Feminicídio,2024-01-01 00:00:00,0.0,NaN,Estadual,1
3829,AL,ANADIA,Feminicídio,2024-01-02 00:00:00,0.0,NaN,Estadual,1
3830,AL,ANADIA,Feminicídio,2024-01-03 00:00:00,0.0,NaN,Estadual,1
3831,AL,ANADIA,Feminicídio,2024-01-04 00:00:00,0.0,NaN,Estadual,1
3832,AL,ANADIA,Feminicídio,2024-01-05 00:00:00,0.0,NaN,Estadual,1
...,...,...,...,...,...,...,...,...
745511,SP,ÓLEO,Tentativa de homicídio,2024-01-08 00:00:00,1.0,NaN,Estadual,1
745512,SP,ÓLEO,Tentativa de homicídio,2024-01-09 00:00:00,1.0,NaN,Estadual,1
745513,SP,ÓLEO,Tentativa de homicídio,2024-01-10 00:00:00,0.0,NaN,Estadual,1
745514,SP,ÓLEO,Tentativa de homicídio,2024-01-11 00:00:00,0.0,NaN,Estadual,1


In [ ]:
abrangencia_estadual[abrangencia_estadual['municipio'] == 'NÃO INFORMADO']

,uf,municipio,evento,data_referencia,total_vitima,ocorrencia,abrangencia,mes
11617,AL,NÃO INFORMADO,Apreensão de Cocaína,2024-01-01 00:00:00,NaN,NaN,Estadual,1
11618,AL,NÃO INFORMADO,Apreensão de Cocaína,2024-01-02 00:00:00,NaN,NaN,Estadual,1
11620,AL,NÃO INFORMADO,Apreensão de Cocaína,2024-01-03 00:00:00,NaN,NaN,Estadual,1
11622,AL,NÃO INFORMADO,Apreensão de Cocaína,2024-01-04 00:00:00,NaN,NaN,Estadual,1
11625,AL,NÃO INFORMADO,Apreensão de Cocaína,2024-01-05 00:00:00,NaN,NaN,Estadual,1
...,...,...,...,...,...,...,...,...
709866,SP,NÃO INFORMADO,Tráfico de drogas,2024-01-08 00:00:00,NaN,3436.0,Estadual,1
709869,SP,NÃO INFORMADO,Tráfico de drogas,2024-01-09 00:00:00,NaN,3114.0,Estadual,1
709870,SP,NÃO INFORMADO,Tráfico de drogas,2024-01-10 00:00:00,NaN,3317.0,Estadual,1
709872,SP,NÃO INFORMADO,Tráfico de drogas,2024-01-11 00:00:00,NaN,2945.0,Estadual,1


#### Decisão de Granularidade
Como iremos realizar uma análise focada em vulnearabilidade social, precisamos que a granularidade dos dados esteja a nível municipal. Dessa forma, um estado com PIB alto não irá mascarar as desigualdades internas daquela região.

In [ ]:
df_crimes = df_crimes[df_crimes['abrangencia'] != 'Polícia Federal'].copy()

#### Junção das colunas 'total_vitima' e 'ocorrencia'


In [20]:
print(df_crimes['evento'].drop_duplicates())

3828                             Feminicídio
3840                        Homicídio doloso
3852         Lesão corporal seguida de morte
3912     Roubo seguido de morte (latrocínio)
3936                Tentativa de feminicídio
3948                  Tentativa de homicídio
11617                   Apreensão de Cocaína
11640                   Apreensão de Maconha
11664                Arma de Fogo Apreendida
11928                                Estupro
11940                  Estupro de vulnerável
11964                       Furto de veículo
12289         Roubo a instituição financeira
12312                         Roubo de carga
12336                       Roubo de veículo
12516                      Tráfico de drogas
Name: evento, dtype: object


In [21]:
print(df_crimes['municipio'].drop_duplicates().count())

3352


In [22]:
print(df_crimes[df_crimes['total_vitima'].isnull()]['evento'].value_counts())

evento
Arma de Fogo Apreendida           1404
Apreensão de Cocaína               156
Apreensão de Maconha               156
Furto de veículo                   156
Roubo a instituição financeira     156
Roubo de carga                     156
Roubo de veículo                   156
Tráfico de drogas                  156
Name: count, dtype: int64


In [23]:
print(df_crimes[df_crimes['ocorrencia'].isnull()]['evento'].value_counts())

evento
Feminicídio                            41724
Homicídio doloso                       41724
Lesão corporal seguida de morte        41724
Roubo seguido de morte (latrocínio)    41724
Tentativa de homicídio                 41724
Tentativa de feminicídio               41700
Apreensão de Cocaína                     156
Apreensão de Maconha                     156
Estupro                                  156
Estupro de vulnerável                    156
Arma de Fogo Apreendida                    3
Name: count, dtype: int64


In [24]:
df_crimes.head()

,uf,municipio,evento,data_referencia,total_vitima,ocorrencia,abrangencia,mes
3828,AL,ANADIA,Feminicídio,2024-01-01 00:00:00,0.0,NaN,Estadual,1
3829,AL,ANADIA,Feminicídio,2024-01-02 00:00:00,0.0,NaN,Estadual,1
3830,AL,ANADIA,Feminicídio,2024-01-03 00:00:00,0.0,NaN,Estadual,1
3831,AL,ANADIA,Feminicídio,2024-01-04 00:00:00,0.0,NaN,Estadual,1
3832,AL,ANADIA,Feminicídio,2024-01-05 00:00:00,0.0,NaN,Estadual,1


In [25]:
df_crimes[(df_crimes['evento'] == 'Apreensão de Cocaína') & (df_crimes['mes'] == 1)]

,uf,municipio,evento,data_referencia,total_vitima,ocorrencia,abrangencia,mes
11617,AL,NÃO INFORMADO,Apreensão de Cocaína,2024-01-01 00:00:00,NaN,NaN,Estadual,1
11618,AL,NÃO INFORMADO,Apreensão de Cocaína,2024-01-02 00:00:00,NaN,NaN,Estadual,1
11620,AL,NÃO INFORMADO,Apreensão de Cocaína,2024-01-03 00:00:00,NaN,NaN,Estadual,1
11622,AL,NÃO INFORMADO,Apreensão de Cocaína,2024-01-04 00:00:00,NaN,NaN,Estadual,1
11625,AL,NÃO INFORMADO,Apreensão de Cocaína,2024-01-05 00:00:00,NaN,NaN,Estadual,1
...,...,...,...,...,...,...,...,...
708966,SP,NÃO INFORMADO,Apreensão de Cocaína,2024-01-08 00:00:00,NaN,NaN,Estadual,1
708969,SP,NÃO INFORMADO,Apreensão de Cocaína,2024-01-09 00:00:00,NaN,NaN,Estadual,1
708970,SP,NÃO INFORMADO,Apreensão de Cocaína,2024-01-10 00:00:00,NaN,NaN,Estadual,1
708973,SP,NÃO INFORMADO,Apreensão de Cocaína,2024-01-11 00:00:00,NaN,NaN,Estadual,1


In [26]:
df_crimes = df_crimes[df_crimes['municipio'] != 'NÃO INFORMADO'].copy()

In [27]:
df_crimes['evento'].drop_duplicates()

3828                            Feminicídio
3840                       Homicídio doloso
3852        Lesão corporal seguida de morte
3912    Roubo seguido de morte (latrocínio)
3936               Tentativa de feminicídio
3948                 Tentativa de homicídio
Name: evento, dtype: object

In [31]:
vitima_nula = df_crimes[df_crimes['total_vitima'].isnull() & df_crimes['ocorrencia'].notnull()].shape[0]
ocorrencia_nula = df_crimes[df_crimes['ocorrencia'].isnull() & df_crimes['total_vitima'].notnull()].shape[0]
ambos_nulos =  df_crimes[df_crimes['ocorrencia'].isnull() & df_crimes['total_vitima'].isnull()].shape[0]
print(f'Soma Total: {vitima_nula+ocorrencia_nula+ambos_nulos}')
print(f'Total de registros no df: {df_crimes.shape[0]}')

Soma Total: 249264
Total de registros no df: 249264


In [32]:
df_crimes.loc[df_crimes['total_vitima'].isnull(),'total_vitima'] = 0
df_crimes.loc[df_crimes['ocorrencia'].isnull(),'ocorrencia'] = 0

In [33]:
df_crimes = df_crimes.copy()
df_crimes.loc[:, 'total_eventos'] = df_crimes['total_vitima'] + df_crimes['ocorrencia']
df_crimes = df_crimes.drop(columns=['total_vitima', 'ocorrencia'])
df_crimes.head()

,uf,municipio,evento,data_referencia,abrangencia,mes,total_eventos
3828,AL,ANADIA,Feminicídio,2024-01-01 00:00:00,Estadual,1,0.0
3829,AL,ANADIA,Feminicídio,2024-01-02 00:00:00,Estadual,1,0.0
3830,AL,ANADIA,Feminicídio,2024-01-03 00:00:00,Estadual,1,0.0
3831,AL,ANADIA,Feminicídio,2024-01-04 00:00:00,Estadual,1,0.0
3832,AL,ANADIA,Feminicídio,2024-01-05 00:00:00,Estadual,1,0.0


In [34]:
tipos = {
    'uf': 'string',
    'municipio': 'string',
    'evento': 'string'
}
df_crimes = df_crimes.astype(tipos)
df_crimes.info()

<class 'pandas.core.frame.DataFrame'>
Index: 249264 entries, 3828 to 745515
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   uf               249264 non-null  string 
 1   municipio        249264 non-null  string 
 2   evento           249264 non-null  string 
 3   data_referencia  249264 non-null  object 
 4   abrangencia      249264 non-null  object 
 5   mes              249264 non-null  int32  
 6   total_eventos    249264 non-null  float64
dtypes: float64(1), int32(1), object(2), string(3)
memory usage: 14.3+ MB


In [35]:
municipios = df_crimes['municipio'].drop_duplicates()
municipios = municipios.str.strip()
municipios

3828                      ANADIA
3960                   ARAPIRACA
4092                     ATALAIA
4224      BARRA DE SANTO ANTÔNIO
4356         BARRA DE SÃO MIGUEL
                   ...          
744856        ÁGUAS DE SÃO PEDRO
744988          ÁLVARES FLORENCE
745120           ÁLVARES MACHADO
745252        ÁLVARO DE CARVALHO
745384                      ÓLEO
Name: municipio, Length: 3351, dtype: string

In [36]:
import sidrapy

In [37]:
df_populacao_estimada = sidrapy.get_table(table_code='6579', territorial_level="6", ibge_territorial_code="all", variable='9324', period='2024') 

df_populacao_estimada.columns = df_populacao_estimada.iloc[0]
df_populacao_estimada = df_populacao_estimada.iloc[1:,[6,4,8]]

df_populacao_estimada.head()

,Município,Valor,Ano
1,Alta Floresta D'Oeste - RO,22853,2024
2,Ariquemes - RO,108573,2024
3,Cabixi - RO,5690,2024
4,Cacoal - RO,97637,2024
5,Cerejeiras - RO,16975,2024


In [38]:
df_populacao_estimada = df_populacao_estimada.rename(columns={"Município":"municipio", "Valor":"populacao","Ano":"ano"})
df_populacao_estimada.head()

,municipio,populacao,ano
1,Alta Floresta D'Oeste - RO,22853,2024
2,Ariquemes - RO,108573,2024
3,Cabixi - RO,5690,2024
4,Cacoal - RO,97637,2024
5,Cerejeiras - RO,16975,2024


In [39]:
estados = df_populacao_estimada['municipio'].str[-2:]
estados = list(estados)
print(estados)

['RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'RO', 'AC', 'AC', 'AC', 'AC', 'AC', 'AC', 'AC', 'AC', 'AC', 'AC', 'AC', 'AC', 'AC', 'AC', 'AC', 'AC', 'AC', 'AC', 'AC', 'AC', 'AC', 'AC', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'AM', 'RR', 'RR', 'RR', 'RR', 'RR', 'RR', 'RR', 'RR', 'RR', 'RR', 'RR', 'RR', 'RR', 'RR', 'RR', 'PA', 'PA', 'PA', 'PA', 'PA', 'PA', 'PA', 'PA', 'PA', 'PA', 'PA', 'PA', 'PA', 'PA', 'PA', 'PA

In [40]:
df_populacao_estimada.insert(1,'uf',estados)
df_populacao_estimada.head()

,municipio,uf,populacao,ano
1,Alta Floresta D'Oeste - RO,RO,22853,2024
2,Ariquemes - RO,RO,108573,2024
3,Cabixi - RO,RO,5690,2024
4,Cacoal - RO,RO,97637,2024
5,Cerejeiras - RO,RO,16975,2024


In [41]:
df_populacao_estimada['municipio'] = df_populacao_estimada['municipio'].str[:-4]
df_populacao_estimada.head()

,municipio,uf,populacao,ano
1,Alta Floresta D'Oeste,RO,22853,2024
2,Ariquemes,RO,108573,2024
3,Cabixi,RO,5690,2024
4,Cacoal,RO,97637,2024
5,Cerejeiras,RO,16975,2024


In [42]:
df_populacao_estimada['municipio'] = df_populacao_estimada['municipio'].str.upper()
df_populacao_estimada['municipio'] = df_populacao_estimada['municipio'].str.strip()
df_populacao_estimada.head()

,municipio,uf,populacao,ano
1,ALTA FLORESTA D'OESTE,RO,22853,2024
2,ARIQUEMES,RO,108573,2024
3,CABIXI,RO,5690,2024
4,CACOAL,RO,97637,2024
5,CEREJEIRAS,RO,16975,2024


In [43]:
df_populacao = df_populacao_estimada.copy()
df_populacao = df_populacao_estimada[df_populacao_estimada['uf'].isin(uf_nordeste_sudeste)]
df_populacao
# num de linhas (uf) diferente do df de crimes

,municipio,uf,populacao,ano
451,AÇAILÂNDIA,MA,110506,2024
452,AFONSO CUNHA,MA,6296,2024
453,ÁGUA DOCE DO MARANHÃO,MA,12446,2024
454,ALCÂNTARA,MA,18774,2024
455,ALDEIAS ALTAS,MA,23782,2024
...,...,...,...,...
3908,VOTORANTIM,SP,132849,2024
3909,VOTUPORANGA,SP,100159,2024
3910,ZACARIAS,SP,2761,2024
3911,CHAVANTES,SP,12411,2024


In [44]:
tipos = {
    'municipio': 'string',
    'uf': 'string',
    'populacao': 'int',
    'ano': 'int'
}

df_populacao = df_populacao.astype(tipos)

df_populacao.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3462 entries, 451 to 3912
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   municipio  3462 non-null   string
 1   uf         3462 non-null   string
 2   populacao  3462 non-null   int64 
 3   ano        3462 non-null   int64 
dtypes: int64(2), string(2)
memory usage: 135.2 KB


In [45]:
municipios_df_crimes = df[['uf', 'municipio']].drop_duplicates()
comparacao = pd.merge(
    municipios_df_crimes, 
    df_populacao, 
    left_on=['uf', 'municipio'], 
    right_on=['uf', 'municipio'], 
    how='left', 
    indicator=True
)
municipios_faltantes = comparacao[comparacao['_merge'] == 'left_only']
print(municipios_faltantes[['uf', 'municipio']])

NameError: name 'df' is not defined